[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/math_module2/blob/main/ch8_convexity.ipynb)

# Chapter 8 — Companion Notebook
## Maxima, Minima, and the Shape of a Function
*where a curve peaks, where it bottoms out, and why some bowls have no false floors*

Three live pictures of the chapter's payoff: classify a critical point with *both* derivative tests while the curve is painted by its curvature, watch chords sit **above** and tangents sit **below** a convex graph, and drop gradient-descent balls into a convex bowl (all reach the one floor) versus a two-valley landscape (they split) — the keystone made visible: *a convex bowl has no false floors.*

> **How to use this notebook.** Run every cell from the top (Shift+Enter). Each widget below is live — drag the sliders and watch the picture respond. Requires `ipywidgets` and `matplotlib`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from ipywidgets import (interact, interactive, fixed, Dropdown, Checkbox,
                        Button, Output, VBox, HBox,
                        FloatSlider, IntSlider, FloatLogSlider)
from IPython.display import HTML, display, Markdown

# Book 2 palette — matches the printed figures in figures/png/
BLUE="#1f4e79"; RED="#c0392b"; GREEN="#2e8b57"; ORANGE="#e08e0b"
PURPLE="#6c3483"; TEAL="#1b9e9e"; GRAY="#7f8c8d"; DARK="#2c3e50"
SHADE="#9fc5e8"; SHADE2="#f4b6ad"; SHADEG="#a9dfbf"; LIGHT="#d6dbdf"

plt.rcParams.update({
    "figure.figsize": (7.2, 4.5), "figure.dpi": 96,
    "font.family": "serif", "mathtext.fontset": "cm", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": LIGHT, "grid.linewidth": 0.7,
    "lines.linewidth": 2.2, "lines.solid_capstyle": "round",
})

def axes(ax=None, title=None):
    '''A clean axes with a light grid; returns the axes.'''
    if ax is None:
        _, ax = plt.subplots()
    if title:
        ax.set_title(title, color=DARK)
    return ax


## 1. Classify the critical point

A **critical point** is where $f'(c)=0$ — a *candidate* for a max or a min, never a verdict on its own. Two tests deliver the verdict:

- **First-derivative test:** watch the sign of $f'$ change across $c$. From $+$ to $-$ is a **maximum**; from $-$ to $+$ is a **minimum**; no change is **neither** (like $x^3$ at $0$).
- **Second-derivative test:** read $f''(c)$. Positive $\Rightarrow$ **minimum** (the curve cups upward), negative $\Rightarrow$ **maximum** (it caps downward), zero $\Rightarrow$ **inconclusive**.

Pick a function. The curve is painted **green** where it is *concave up* ($f''>0$, a cup) and **red** where it is *concave down* ($f''<0$, a cap); hollow dots mark the **inflection points** where the two meet. Each critical point is flagged with what *both* tests conclude — and you can confirm they agree. *(Figures 8.2, 8.3.)*

In [ ]:
# (f, f', f'') for each menu entry, with a plotting range.
_W1 = {
    'f(x) = x^3 - 3x': dict(
        f=lambda x: x**3 - 3*x, fp=lambda x: 3*x**2 - 3,
        fpp=lambda x: 6*x, lo=-2.4, hi=2.4),
    'f(x) = x^4 - 2x^2': dict(
        f=lambda x: x**4 - 2*x**2, fp=lambda x: 4*x**3 - 4*x,
        fpp=lambda x: 12*x**2 - 4, lo=-1.8, hi=1.8),
    'f(x) = x^3 (saddle slope)': dict(
        f=lambda x: x**3, fp=lambda x: 3*x**2,
        fpp=lambda x: 6*x, lo=-1.6, hi=1.6),
    'f(x) = sin(x)': dict(
        f=lambda x: np.sin(x), fp=lambda x: np.cos(x),
        fpp=lambda x: -np.sin(x), lo=-2*np.pi, hi=2*np.pi),
}

def _sign_changes(g, xs):
    '''Indices i where g(xs) changes sign between i and i+1.'''
    s = np.sign(g(xs))
    s[s == 0] = 1                 # treat exact zeros as a side
    return np.where(np.diff(s) != 0)[0]

def _classify(curve='f(x) = x^3 - 3x'):
    d = _W1[curve]
    f, fp, fpp, lo, hi = d['f'], d['fp'], d['fpp'], d['lo'], d['hi']
    xs = np.linspace(lo, hi, 1600)
    ys = f(xs)

    fig, ax = plt.subplots()
    # Paint the curve by the sign of f'' (concavity).
    up = fpp(xs) > 0
    bnd = [0] + list(np.where(np.diff(up.astype(int)) != 0)[0] + 1)
    bnd += [len(xs)]
    lab_up = lab_dn = True
    for i in range(len(bnd) - 1):
        s, e = bnd[i], bnd[i + 1]
        if e - s < 2:
            continue
        cu = up[s]
        col = GREEN if cu else RED
        lab = None
        if cu and lab_up:
            lab, lab_up = "concave up  (f'' > 0)", False
        elif (not cu) and lab_dn:
            lab, lab_dn = "concave down  (f'' < 0)", False
        ax.plot(xs[s:e+1], ys[s:e+1], color=col, lw=3.0, label=lab,
                zorder=2)
    # Inflection points: where f'' changes sign.
    inf = _sign_changes(fpp, xs)
    if inf.size:
        xi = 0.5 * (xs[inf] + xs[inf + 1])
        ax.scatter(xi, f(xi), s=70, facecolors='white',
                   edgecolors=DARK, linewidths=1.8, zorder=6,
                   label='inflection  (f\u2032\u2032 = 0)')
    # Critical points: where f' changes sign OR just touches zero.
    crit_i = _sign_changes(fp, xs)
    xc = list(0.5 * (xs[crit_i] + xs[crit_i + 1]))
    # Also catch f' = 0 with no sign change (e.g. x^3 at 0): local min of |f'|.
    near = np.where(np.abs(fp(xs)) < 1e-2)[0]
    for k in near:
        x0 = xs[k]
        if all(abs(x0 - c) > 0.15 for c in xc):
            xc.append(x0)
    xc = sorted(xc)

    verdicts = []
    h = 1e-3
    for c in xc:
        ax.scatter([c], [f(c)], s=95, color=DARK, zorder=7)
        # First-derivative test: sign of f' just left / right.
        sl, sr = fp(c - 5*h), fp(c + 5*h)
        if sl > 0 and sr < 0:
            fdt = 'max'
        elif sl < 0 and sr > 0:
            fdt = 'min'
        else:
            fdt = 'neither'
        # Second-derivative test: sign of f''(c).
        v = fpp(c)
        if v > 1e-6:
            sdt = 'min'
        elif v < -1e-6:
            sdt = 'max'
        else:
            sdt = 'inconclusive (f\u2032\u2032 = 0)'
        verdicts.append((c, fdt, sdt))
        ax.axvline(c, color=GRAY, lw=0.8, ls=':')
    ax.axhline(0, color=GRAY, lw=0.8, ls=':')
    ax.set_xlim(lo, hi)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title('Curvature painted on; critical points classified',
                 color=DARK)
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    plt.show()

    # Print both verdicts so the reader sees them agree.
    if not verdicts:
        print('No critical points (f\u2032 \u2260 0) in this window.')
    else:
        print(f'Critical points of  {curve}:')
        for c, fdt, sdt in verdicts:
            agree = '\u2713 agree' if fdt == sdt else (
                '(2nd test inconclusive \u2014 trust the 1st)')
            print(f"  x \u2248 {c:+.3f}:  1st-deriv test \u2192 {fdt:<8}"
                  f"  2nd-deriv test \u2192 {sdt:<24} {agree}")

interact(_classify,
         curve=Dropdown(options=list(_W1), description='function'));


## 2. Convexity checker: chords above, tangents below

Two of the **four faces of convexity** are pictures you can test by eye:

- **Chord above graph** — the straight line between any two points on the graph lies *on or above* the curve.
- **Tangent below graph** — the graph lies *on or above* every one of its tangent lines.

A **convex** function ($f''\ge 0$ everywhere) passes both, always. A non-convex one fails somewhere. Pick a function and slide the **seed** to draw fresh random chords (**orange**) and tangents (**red**). Each gets a $\checkmark$ if it lands on the correct side and a $\times$ if it doesn't — so the convex curves come up all checks, and the wavy one gives itself away. *(Faces 1 and 2.)*

In [ ]:
_W2 = {
    'f(x) = x^2  (convex)':  dict(
        f=lambda x: x**2, fp=lambda x: 2*x,
        lo=-3.0, hi=3.0, convex=True),
    'f(x) = e^x  (convex)':  dict(
        f=lambda x: np.exp(x), fp=lambda x: np.exp(x),
        lo=-2.2, hi=1.8, convex=True),
    'f(x) = x^4  (convex)':  dict(
        f=lambda x: x**4, fp=lambda x: 4*x**3,
        lo=-1.7, hi=1.7, convex=True),
    'f(x) = x^3 - 3x  (NOT convex)': dict(
        f=lambda x: x**3 - 3*x, fp=lambda x: 3*x**2 - 3,
        lo=-2.4, hi=2.4, convex=False),
    'f(x) = sin(x)  (NOT convex)': dict(
        f=lambda x: np.sin(x), fp=lambda x: np.cos(x),
        lo=-np.pi, hi=np.pi, convex=False),
}

def _convexity(curve='f(x) = x^2  (convex)', seed=0, n_lines=3):
    d = _W2[curve]
    f, fp, lo, hi = d['f'], d['fp'], d['lo'], d['hi']
    rng = np.random.default_rng(seed)
    xs = np.linspace(lo, hi, 800)
    ys = f(xs)

    fig, ax = plt.subplots()
    ax.plot(xs, ys, color=BLUE, lw=2.6, label='$f(x)$', zorder=3)

    chord_ok = tang_ok = 0
    lab_c = lab_t = True
    span = hi - lo
    for _ in range(n_lines):
        # --- a random chord: should lie ABOVE a convex graph ---
        a, b = np.sort(rng.uniform(lo, hi, 2))
        if b - a < 0.3 * span:        # keep chords visible
            b = min(hi, a + 0.45 * span)
        t = np.linspace(0, 1, 60)
        cx = a + t * (b - a)
        chord = f(a) + t * (f(b) - f(a))      # the straight line
        above = np.all(chord >= f(cx) - 1e-9)  # chord on/above curve
        chord_ok += int(above)
        lab = ('chord (should be above)' if lab_c else None)
        lab_c = False
        ax.plot(cx, chord, color=ORANGE, lw=1.8, zorder=4, label=lab)
        mark = '$\\checkmark$' if above else '$\\times$'
        mc = GREEN if above else RED
        ax.annotate(mark, xy=(cx[30], chord[30]), color=mc,
                    fontsize=15, ha='center', va='bottom', zorder=8)

        # --- a random tangent: should lie BELOW a convex graph ---
        x0 = rng.uniform(lo + 0.1*span, hi - 0.1*span)
        tx = np.linspace(lo, hi, 60)
        tang = f(x0) + fp(x0) * (tx - x0)      # the tangent line
        below = np.all(tang <= f(tx) + 1e-9)   # tangent on/below curve
        tang_ok += int(below)
        lab = ('tangent (should be below)' if lab_t else None)
        lab_t = False
        ax.plot(tx, tang, color=RED, lw=1.6, ls='--', zorder=2,
                label=lab)
        mark = '$\\checkmark$' if below else '$\\times$'
        mc = GREEN if below else RED
        ax.scatter([x0], [f(x0)], s=30, color=RED, zorder=6)
        ax.annotate(mark, xy=(tx[5], tang[5]), color=mc,
                    fontsize=15, ha='center', va='top', zorder=8)

    pad = 0.12 * (np.nanmax(ys) - np.nanmin(ys) + 1e-9)
    ax.set_ylim(np.nanmin(ys) - 3*pad, np.nanmax(ys) + 3*pad)
    ax.set_xlim(lo, hi)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    verdict = 'convex \u2014 all on the right side' if d['convex'] \
        else 'not convex \u2014 some land wrong'
    ax.set_title(verdict, color=(GREEN if d['convex'] else RED))
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    plt.show()

    print(f'{curve}')
    print(f'  chords above curve : {chord_ok}/{n_lines}')
    print(f'  tangents below curve: {tang_ok}/{n_lines}')
    if d['convex']:
        print('  \u2713 convex: every chord is above and every '
              'tangent is below (Faces 1 & 2).')
    else:
        print('  \u2717 non-convex: at least one chord dips below or '
              'a tangent pokes above.')

interact(_convexity,
         curve=Dropdown(options=list(_W2), description='function'),
         seed=IntSlider(value=0, min=0, max=20, step=1,
                        description='seed'),
         n_lines=IntSlider(value=3, min=1, max=5, step=1,
                           description='# lines'));


## 3. Convex vs non-convex descent

Here is the keystone of the whole book, made visible. Gradient descent only ever steps **downhill**: $x \leftarrow x - \eta\, f'(x)$, with **learning rate** $\eta$.

- **Left — a convex bowl** $f(x)=x^2$. By the theorem, the one point where $f'=0$ is the *global* minimum, and there is nothing else to fall into. Balls dropped from *anywhere* (**purple**) all roll to the same single floor. **No false floors.**
- **Right — a two-valley landscape** $f(x)=x^4-3x^2+x$. This is *non-convex*: two separate dips of different depth. Balls (**teal**) **split** — each falls into whichever valley is nearest and stops, even when a deeper one waits across the ridge. This is exactly the Chapter 9 lab where the starting point decided the outcome.

Slide the **learning rate** and the **number of starts**. Watch the left panel always agree and the right panel disagree — that disagreement *is* non-convexity, and it is why "is your problem convex?" was, for decades, the first question in optimization.

In [ ]:
def _descend(f, fp, x0, eta, steps=60):
    '''Return the gradient-descent path from x0.'''
    xs = [x0]
    x = x0
    for _ in range(steps):
        x = x - eta * fp(x)
        if not np.isfinite(x) or abs(x) > 1e3:   # diverged
            break
        xs.append(x)
    return np.array(xs)

# Left: convex bowl.   Right: two-valley landscape.
_fL  = lambda x: x**2;            _fLp = lambda x: 2*x
_fR  = lambda x: x**4 - 3*x**2 + x
_fRp = lambda x: 4*x**3 - 6*x + 1

def _split_demo(eta=0.10, n_starts=7):
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(11.5, 4.6))

    # ----- Left panel: convex bowl, every ball -> one floor -----
    xs = np.linspace(-3.2, 3.2, 500)
    axL.plot(xs, _fL(xs), color=BLUE, lw=2.4, zorder=2)
    starts = np.linspace(-3.0, 3.0, n_starts)
    ends = []
    for x0 in starts:
        path = _descend(_fL, _fLp, x0, eta)
        axL.plot(path, _fL(path), color=PURPLE, lw=1.0, alpha=0.5,
                 zorder=3)
        axL.scatter([x0], [_fL(x0)], s=28, color=PURPLE, alpha=0.6,
                    zorder=4)
        axL.scatter([path[-1]], [_fL(path[-1])], s=70, color=PURPLE,
                    edgecolors='white', linewidths=1.2, zorder=6)
        ends.append(path[-1])
    axL.scatter([0], [0], marker='*', s=260, color=GREEN,
                edgecolors=DARK, linewidths=1.0, zorder=7,
                label='the one floor')
    axL.set_title('Convex bowl  $x^2$  \u2014 no false floors',
                  color=GREEN)
    axL.set_xlabel('$x$'); axL.set_ylabel('$f(x)$')
    axL.legend(loc='upper center', fontsize=9, framealpha=0.9)
    n_floors_L = len(np.unique(np.round(ends, 1)))

    # ----- Right panel: two valleys, balls split -----
    xr = np.linspace(-2.1, 2.0, 600)
    axR.plot(xr, _fR(xr), color=BLUE, lw=2.4, zorder=2)
    ends = []
    for x0 in np.linspace(-1.9, 1.8, n_starts):
        path = _descend(_fR, _fRp, x0, eta)
        axR.plot(path, _fR(path), color=TEAL, lw=1.0, alpha=0.55,
                 zorder=3)
        axR.scatter([x0], [_fR(x0)], s=28, color=TEAL, alpha=0.6,
                    zorder=4)
        axR.scatter([path[-1]], [_fR(path[-1])], s=70, color=TEAL,
                    edgecolors='white', linewidths=1.2, zorder=6)
        ends.append(path[-1])
    # The two true minima of x^4 - 3x^2 + x (roots of 4x^3 - 6x + 1).
    roots = np.roots([4, 0, -6, 1])
    mins = sorted([r.real for r in roots
                   if abs(r.imag) < 1e-9 and (12*r.real**2 - 6) > 0])
    axR.scatter(mins, [_fR(m) for m in mins], marker='*', s=220,
                color=ORANGE, edgecolors=DARK, linewidths=1.0,
                zorder=7, label='two separate minima')
    axR.set_title('Two valleys  $x^4-3x^2+x$  \u2014 balls split',
                  color=RED)
    axR.set_xlabel('$x$'); axR.set_ylabel('$f(x)$')
    axR.legend(loc='upper center', fontsize=9, framealpha=0.9)
    n_floors_R = len(np.unique(np.round(ends, 1)))

    fig.suptitle(f'learning rate \u03b7 = {eta:.3f}', color=DARK,
                 fontsize=12)
    fig.tight_layout()
    plt.show()

    print(f'Convex bowl : balls landed on {n_floors_L} distinct floor'
          f'(s) \u2014 convexity guarantees just one.')
    print(f'Two valleys : balls landed on {n_floors_R} distinct floor'
          f'(s) \u2014 the outcome depends on where you start.')

interact(_split_demo,
         eta=FloatSlider(value=0.10, min=0.01, max=0.18, step=0.01,
                         description='learn rate', readout_format='.3f'),
         n_starts=IntSlider(value=7, min=2, max=12, step=1,
                            description='# starts'));


---

*Three readings of one chapter. The first sorts critical points with both tests while the curvature is painted on. The second tests convexity by eye — chords above, tangents below. The third makes the keystone unmistakable: on a convex bowl every downhill path ends at the same floor, while a non-convex landscape sends them to different valleys. That single difference — **no false floors** — is the line between optimization that is guaranteed to work and the non-convex world of Chapter 9, where training a model is part science and part art.*